### Imports

In [16]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from BayesCalculator import BayesCalculator

In [17]:
path="NSE Data"

In [18]:
df = pd.read_csv(path, header=[0,1,2])
df.columns = df.columns.get_level_values(0)
display(df.head(3))

,Price,Close,High,Low,Open,Volume
0,2007-09-17,4494.649902,4549.049805,4482.850098,4518.450195,0
1,2007-09-18,4546.200195,4551.799805,4481.549805,4494.100098,0
2,2007-09-19,4732.350098,4739.000000,4550.250000,4550.250000,0


### Pulling usefull data

In [19]:
df["Returns"] = df["Close"].pct_change()
df["Mean_30"]= df["Close"].rolling(window=30).mean()
df["Std_30"] = df["Close"].rolling(window=30).std()

df= df.dropna()
display(df.head(5))

,Price,Close,High,Low,Open,Volume,Returns,Mean_30,Std_30
29,2007-10-29,5905.899902,5922.500000,5708.899902,5708.899902,0,0.035705,5215.598324,355.079530
30,2007-10-30,5868.750000,5976.000000,5833.899902,5917.549805,0,-0.006290,5261.401660,347.417310
31,2007-10-31,5900.649902,5952.250000,5861.299805,5868.899902,0,0.005436,5306.549984,339.179480
32,2007-11-01,5866.450195,6011.950195,5837.200195,5903.799805,0,-0.005796,5344.353320,336.162360
33,2007-11-02,5932.399902,5944.750000,5714.250000,5854.850098,0,0.011242,5383.848324,333.217208


In [20]:
close = df["Close"].to_numpy()
returns= df["Returns"].to_numpy()
mean30= df["Mean_30"].to_numpy()
std30= df["Std_30"].to_numpy()

### Setting up Transition Matrices

In [21]:
'''
Today         Tomorrow  Bull    Bear 
Bull                    0.95    0.05
Bear                    0.10    0.90
'''



transition_matrix= np.array(
    [
        [0.95,0.05],            #If today is Bull, 95% chance it stays Bull tomorrow
        [0.10,0.90]             # If today is Bear, 90% chance it stays Bear tomorrow
    ]
)

### Setting Up Initial Probablities or Beliefs

In [22]:
initial_belief= np.array([0.5,0.5])
current_belief= initial_belief
belief_list =[]     # For tracking all the probablities as the things move forward.

In [23]:
belief_list.append(current_belief)

### Updating Beliefs Function

In [24]:
def update(current_belief, transition_matrix, returns, mean, std):
    #Predict Belief based on yesterday's probablity and transition matrix.
    predicted_belief = np.dot(current_belief, transition_matrix) #Gives us predicted bull and predicted bear
    p_bull_prior = predicted_belief[0]
    p_bear_prior = predicted_belief[1]

    #Bull
    like_bull= norm.pdf(returns,loc=mean, scale=std)

    #Bear
    like_bear= norm.pdf(returns, -abs(mean), std*1.5)

    epsilon = 1e-300
    like_bull = max(like_bull, epsilon)
    like_bear = max(like_bear, epsilon)
    #Bayes Theorem
    calc = BayesCalculator(
        p_a=p_bull_prior,
        p_not_a=p_bear_prior,
        p_b_given_a=like_bull,
        p_b_given_not_a=like_bear
    )
    posterior_bull = calc.compute()
    posterior_bear = 1.0 - posterior_bull

    if np.isnan(posterior_bull) or np.isnan(posterior_bear):
        return predicted_belief.tolist()

    return [posterior_bull,posterior_bear]

### Puting the code in use

In [25]:
n_days=len(returns)

for i in range(1, n_days):
    current_belief= update(current_belief=current_belief, transition_matrix=transition_matrix, returns= returns[i], mean=mean30[i], std=std30[i])
    belief_list.append(current_belief)

In [27]:
display(current_belief)

[np.float64(0.48500631664), np.float64(0.51499368336)]

In [33]:
if current_belief[0]>current_belief[1]:
    print(f"Its a Bull Market with the Probability of {current_belief[0]*100: .2f}%")
else:
    print(f"Its a Bear Market with the Proability of {current_belief[1]*100: .2f}%")

Its a Bear Market with the Proability of  51.50%
